In [1]:
import av
import torch
import numpy as np
from transformers import AutoProcessor, LlavaNextVideoForConditionalGeneration
import cv2

In [2]:
# Define model ID
model_id = "llava-hf/LLaVA-NeXT-Video-7B-hf"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
# Load the processor and model
model = LlavaNextVideoForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    low_cpu_mem_usage=True,
    load_in_4bit=True,
    # attn_implementation="flash_attention_2",
    )

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
processor = AutoProcessor.from_pretrained(model_id)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 5ef03b69-f76e-4d94-885b-cdbee3181b73)')' thrown while requesting HEAD https://huggingface.co/llava-hf/LLaVA-NeXT-Video-7B-hf/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


In [11]:
# Function to read video frames using OpenCV
def read_video_opencv(video_path, num_frames=8):
    '''
    Decode video frames using OpenCV.
    Args:
        video_path (str): Path to the video file.
        num_frames (int): Number of frames to decode.
    Returns:
        result (np.ndarray): Array of decoded frames of shape (num_frames, height, width, 3).
    '''
    frames = []
    index = 0
    video = cv2.VideoCapture(video_path)
    fps = int(video.get(cv2.CAP_PROP_FPS))
    total_num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.arange(0, total_num_frames, total_num_frames / num_frames).astype(int)
    while video.isOpened():
        success, frame = video.read()
        if index in indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        if success:
            index += 1
        if index >= total_num_frames:
            break

    video.release()

    if not frames:
        raise ValueError("No frames were read from the video.")

    return np.stack(frames)


In [9]:
conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "text", "text": "What is abnormal action in this video?",
            },
            {
                "type": "video"
            }
        ]
    }
]

prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

In [28]:
# video_path = "D:/6. Datasets/Fall-Dataset/Cut/C00_010_0004-C1.mp4"
# video_path = "D:/6. Datasets/Fight-Dataset/C00_083_0003-C1.mp4"
video_path = "D:/6. Datasets/Fight-Dataset/fight-video-2.mp4"
clip = read_video_opencv(video_path, num_frames=8)

inputs = processor(text=prompt, videos=clip, padding=True, return_tensors="pt").to(device)

In [29]:
outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(outputs[0], skip_special_tokens=True))

USER: 
What is abnormal action in this video? ASSISTANT: The video shows a group of people engaging in a physical altercation, with one person lying on the ground and another person standing over them. This is an abnormal action as it is not a typical or safe way to resolve conflicts. The situation appears to be escalating and could potentially lead to serious injury.
